# Demo 6 — Equation Solving Attacks
## Algebraic Model Inversion on Linear Classifiers

**Threat Model:** White-box, analytical

**Goal:** Recover the model's weight vector **w** and bias **b** exactly
by treating the model as a **system of linear equations**, then craft
the *minimum-norm* adversarial perturbation analytically.

**Formula for linear model:** `f(x) = w·x + b`

If the attacker knows `n+1` query scores (where `n` = feature dimension):

$\mathbf{X}\mathbf{w} = \mathbf{s}$ → solved via least-squares

In [ ]:
# Install
# !pip install numpy scikit-learn matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_classification
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
np.random.seed(42)

## Step 1 — Train a Victim Linear Classifier

We use a **2D dataset** for easy boundary visualisation.

In [ ]:
X, y = make_classification(n_samples=500, n_features=2, n_informative=2,
                            n_redundant=0, n_clusters_per_class=1, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)
victim = LogisticRegression(random_state=42, max_iter=1000)
victim.fit(X_train_s, y_train)
w_true = victim.coef_[0]; b_true = victim.intercept_[0]
print(f'Victim accuracy: {victim.score(X_test_s, y_test)*100:.1f}%')
print(f'True weights: w={w_true}, b={b_true:.4f}')

## Step 2 — Equation Solving: Recover w and b

The attacker sends `n+1` probe inputs to the API and receives **raw logit scores**.
They set up the system $Aw = s$ and solve: `w = A^{-1} s`.

In [ ]:
n_features   = X_train_s.shape[1]
probe_inputs = np.eye(n_features+1)[:n_features+1, :n_features]
probe_inputs = probe_inputs + np.random.randn(*probe_inputs.shape) * 0.01
probe_scores = victim.decision_function(probe_inputs)
A = np.hstack([probe_inputs, np.ones((len(probe_inputs), 1))])
params, _, _, _ = np.linalg.lstsq(A, probe_scores, rcond=None)
w_recovered = params[:n_features]; b_recovered = params[-1]
print(f'True      : w={w_true}, b={b_true:.4f}')
print(f'Recovered : w={w_recovered}, b={b_recovered:.4f}')
print(f'L2 error  : {np.linalg.norm(w_true - w_recovered):.8f}')

## Step 3 — Craft Minimal Adversarial Examples Analytically

Using recovered weights, compute the **minimum L2 perturbation** to cross
the decision boundary:

$\delta^* = -\frac{\mathbf{w}^T\mathbf{x} + b}{\|\mathbf{w}\|^2} \cdot \mathbf{w}$

In [ ]:
def craft_adversarial(x, w, b):
    margin = (w @ x + b) / (np.linalg.norm(w) ** 2)
    return x - margin * w

original, adv_examples = [], []
for xi, yi in zip(X_test_s, y_test):
    if yi == 1:
        original.append(xi)
        adv_examples.append(craft_adversarial(xi, w_recovered, b_recovered))
original     = np.array(original)
adv_examples = np.array(adv_examples)
success_rate = (victim.predict(adv_examples) == 0).mean()
print(f'Attack success rate (flipped to class 0): {success_rate*100:.1f}%')
print(f'Mean L2 perturbation: {np.linalg.norm(adv_examples - original, axis=1).mean():.4f}')

In [ ]:
def plot_boundary(ax, model, X, y, title):
    h = 0.05
    x_min, x_max = X[:, 0].min()-1, X[:, 0].max()+1
    y_min, y_max = X[:, 1].min()-1, X[:, 1].max()+1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.2, cmap='RdBu')
    ax.scatter(X[y==0, 0], X[y==0, 1], c='blue', s=20, label='Class 0')
    ax.scatter(X[y==1, 0], X[y==1, 1], c='red',  s=20, label='Class 1')
    ax.set_title(title, fontweight='bold'); ax.legend(fontsize=8)

fig_before, ax_before = plt.subplots(figsize=(6, 5))
plot_boundary(ax_before, victim, X_test_s, y_test, 'Before Attack: Clean Boundary')
plt.savefig('06_before_attack.png', dpi=150)
plt.close()

fig_mid, ax_mid = plt.subplots(figsize=(6, 5))
dummy_model = LogisticRegression(); dummy_model.coef_ = np.array([w_recovered]); dummy_model.intercept_ = np.array([b_recovered]); dummy_model.classes_ = np.array([0, 1])
plot_boundary(ax_mid, dummy_model, X_test_s, y_test, 'In Process: Recovered Boundary')
plt.savefig('06_in_process_attack.png', dpi=150)
plt.close()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
plot_boundary(axes[0], victim, X_test_s, y_test, 'Clean Decision Boundary')
plot_boundary(axes[1], victim, X_test_s, y_test, 'Equation Solving Adversarial Examples')
for o, a in zip(original[:20], adv_examples[:20]):
    axes[1].annotate('', xy=a, xytext=o, arrowprops=dict(arrowstyle='->', color='orange', lw=1.5))
axes[1].scatter(adv_examples[:,0], adv_examples[:,1], c='orange', marker='x', s=60,
                zorder=5, label='Adversarial')
axes[1].legend(fontsize=8)
plt.suptitle('Equation Solving Attack', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('06_after_attack.png', dpi=150)
plt.show()

## Summary

Equation solving attacks show that **linear models can be fully compromised**
with just `n+1` queries. Once `w` and `b` are known:
- The attacker computes the **exact optimal adversarial perturbation** analytically
- Near-100% success rate with **minimal perturbation**

**Defenses:** Output rounding, differential privacy noise, score perturbation.